## 编辑距离（Edit Distance）
是一种衡量两个字符串之间相似度的指标，它表示将一个字符串转换为另一个字符串所需的最少单字符编辑操作次数（如插入、删除或替换一个字符）。

In [2]:
# 直接使用库函数

import Levenshtein

str1 = '你好NLP'
str2 = '你好CV'

distance = Levenshtein.distance(str1, str2)
print('编辑距离：', distance)

编辑距离： 3


In [3]:
# 自己实现

def minDistance(word1, word2):
    n1 = len(word1)
    n2 = len(word2)
    dp = [[0] * (n2 + 1) for _ in range(n1 + 1)]
    # 第一行
    for j in range(1, n2 + 1):
        dp[0][j] = dp[0][j-1] + 1
    # 第一列
    for i in range(1, n1 + 1):
        dp[i][0] = dp[i-1][0] + 1
    for i in range(1, n1 + 1):
        for j in range(1, n2 + 1):
            if word1[i-1] == word2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = min(dp[i][j-1], dp[i-1][j], dp[i-1][j-1] ) + 1
    return dp[-1][-1]

str1 = '你好NLP'
str2 = '你好CV'
distance = minDistance(str1, str2)
print('编辑距离：', distance)

编辑距离： 3


### SIMHASH算法
是一种用于计算文本相似度的方法，通过将文本转换为固定长度的哈希值（SimHash值），然后计算两个文本的SimHash值之间的汉明距离（Hamming distance）来衡量它们的相似度。</br>
汉明距离越小，两个文本的相似度越高。</br>
### 汉明距离
汉明距离（Hamming Distance）是指两个等长字符串在相同位置上不同字符的个数。

In [4]:
import jieba

def word_cut(text):
    cuts_generator = jieba.cut(text, cut_all=False)
    return [word for word in cuts_generator if len(word) > 1]

def del_word(words, del_words):
    return [word for word in words if word not in del_words]

def prepare(text):
    with open('./data/stopwords.dat', 'r') as fr:
        stop_words = [x.strip() for x in fr.readlines()]

    special_words = ['，', '。', ':', '：', '&', '__', '【', '】', '(', ')', '......', '！', '$', '\t', '~']

    words = word_cut(text)
    words = del_word(words, stop_words)
    words = del_word(words, special_words)
    return words

In [6]:
from simhash import Simhash

def text_to_simhash(text):
    # 对文本进行分词
    sim_word = prepare(text.strip())
    # 计算SimHash值
    sh = Simhash(sim_word)
    return sh

def calculate_distance(text1, text2):
    # 计算两个文本的SimHash值
    simhash1 = text_to_simhash(text1)
    simhash2 = text_to_simhash(text2)
    # 计算汉明距离
    hamming_distance = simhash1.distance(simhash2)
    return hamming_distance

if __name__ == "__main__":
    text1 = "simhash是一种计算文本相似度的方法，通过将文本转换为固定长度的哈希值，然后计算两个文本的SimHash值之间的汉明距离来衡量它们的相似度"
    text2 = "simhash是一种用于计算文本相似度的方法，将文本转换为固定长度的hash值，再计算两个文本的SimHash值间的Hamming distance衡量它们的相似度"
    similarity = calculate_distance(text1, text2)
    print("距离：", similarity)

Building prefix dict from the default dictionary ...
Dumping model to file cache /var/folders/y3/spkvf7mn69j10zpv19hmb9300000gn/T/jieba.cache
Loading model cost 0.432 seconds.
Prefix dict has been built successfully.


距离： 9


### word2vec解决语义级别的短文本相似问题

In [7]:
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import jieba
import numpy as np

def get_word2vec():
    # 加载模型
    w2v_model = Word2Vec.load('./model/w2v_skip_gram.model')
    return w2v_model

def text_to_vector(text):
    word_vectors = []
    for word in text:
        if word in w2v_model.wv:
            word_vectors.append(w2v_model.wv[word])
    return np.mean(word_vectors, axis=0)

def get_similar(text1, text2):
    vector1 = text_to_vector(text1)
    vector2 = text_to_vector(text2)
    similarity = cosine_similarity([vector1], [vector2])[0][0]
    return similarity

w2v_model = get_word2vec()

text1 = '这是一个关于计算机的问题'
text2 = '这个问题与计算机有关'
print(f'【{text1}】与【{text2}】之间的相似度为：{get_similar(prepare(text1), prepare(text2))}')

text1 = '淘宝是中国最大的电商平台'
text2 = '拼多多是电商平台的后起之秀'
print(f'【{text1}】与【{text2}】之间的相似度为：{get_similar(prepare(text1), prepare(text2))}')

text1 = '今年的考研人数飙升'
text2 = '00后正在整顿职场'
print(f'【{text1}】与【{text2}】之间的相似度为：{get_similar(prepare(text1), prepare(text2))}')

text1 = '组员们一致认为这次的项目能大获全胜'
text2 = '老刘正在村子里散步'
print(f'【{text1}】与【{text2}】之间的相似度为：{get_similar(prepare(text1), prepare(text2))}')

/Users/hexialong/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


【这是一个关于计算机的问题】与【这个问题与计算机有关】之间的相似度为：0.9157022833824158
【淘宝是中国最大的电商平台】与【拼多多是电商平台的后起之秀】之间的相似度为：0.9036420583724976
【今年的考研人数飙升】与【00后正在整顿职场】之间的相似度为：0.6656561493873596
【组员们一致认为这次的项目能大获全胜】与【老刘正在村子里散步】之间的相似度为：0.41071876883506775


In [68]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer,AdamW
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split

/Users/wenyun_kang/anaconda3/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/Users/wenyun_kang/anaconda3/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN3c106detail19maybe_wrap_dim_slowIxEET_S2_S2_b
  Referenced from: <E03EDA44-89AE-3115-9796-62BA9E0E2EDE> /Users/wenyun_kang/anaconda3/lib/python3.11/site-packages/torchvision/image.so
  Expected in:     <F2FE5CF8-5B5B-3FAD-ADF8-C77D90F49FC9> /Users/wenyun_kang/anaconda3/lib/python3.11/site-packages/torch/lib/libc10.dylib'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [69]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2", force_download=True, resume_download=False)

OSError: Consistency check failed: file should be of size 548105171 but has size 7519472 (model.safetensors).
We are sorry for the inconvenience. Please retry download and pass `force_download=True, resume_download=False` as argument.
If the issue persists, please let us know by opening an issue on https://github.com/huggingface/huggingface_hub.